# HCTiGT: The First Step
## You can train a transformer! 🌱

This is a small, real experiment. You'll take **GPT-2**, show it a few stories, update its weights, and see what happens.

### 🛑 STOP! Switch to a GPU first!

**Runtime → Change runtime type → T4 GPU → Save.**

Click ▶ next to a cell, or press **Shift + Enter**. Try these two cells first!


In [ ]:
print('Hello, world!')


**And this one!**


In [ ]:
import torch
assert torch.cuda.is_available(), '🛑 Switch to T4 GPU: Runtime → Change runtime type → T4 GPU → Save.'
print('GPU:', torch.cuda.get_device_name(0))


Nice! Now run the cells below, from top to bottom. **Part A** runs unchanged; in **Part B**, choose your own experiment.

We're *fine-tuning* a pretrained transformer, not training GPT-2 from scratch.


## Part A — Just run it

First, bring in the tools. Colab already includes PyTorch. The following cell installs Hugging Face Transformers.


In [ ]:
%pip -q install "transformers>=4.44,<5" "requests>=2.31,<3"


In [ ]:
import random
from functools import lru_cache

import matplotlib.pyplot as plt
import requests
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)
if DEVICE.type == "cpu":
    print("A GPU is recommended; the same code on CPU may take much longer.")


### The training examples

The stories come from [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories). We fetch just a handful using the Hugging Face dataset-viewer API, rather than downloading a huge dataset.

There's also a **math track**: short addition examples generated right here in Python. We'll try that in Part B.


In [ ]:
@lru_cache(maxsize=8)
def training_examples(track, count):
    if not 8 <= count <= 100:
        raise ValueError("Choose between 8 and 100 examples.")
    if track == "stories":
        response = requests.get(
            "https://datasets-server.huggingface.co/rows",
            params={
                "dataset": "roneneldan/TinyStories",
                "config": "default",
                "split": "train",
                "offset": 0,
                "length": count,
            },
            timeout=90,
        )
        response.raise_for_status()
        examples = [item["row"]["text"].strip() for item in response.json()["rows"]]
        if len(examples) != count:
            raise RuntimeError("TinyStories returned fewer examples than requested.")
        return examples
    if track == "math":
        rng = random.Random(42)
        pairs = [(a, b) for a in range(20) for b in range(20)]
        rng.shuffle(pairs)
        return [f"Question: {a} + {b}\nAnswer: {a+b}\n" for a, b in pairs[:count]]
    raise ValueError('Track must be "stories" or "math".')


def prompt_for(track):
    return (
        "Once upon a time, there was a little dragon"
        if track == "stories"
        else "Question: 7 + 5\nAnswer:"
    )


### The training loop

**Examples → tokens → predictions → loss → gradients → changed weights.**

You don't need to understand every line to run it. We load the same original GPT-2 every time so a new experiment starts fresh. We'll measure loss on **four training examples** before and after (not on a held-out test set).


In [ ]:
MODEL_NAME = "openai-community/gpt2"  # GPT-2 small (124M parameters)
TOKEN_LIMIT = 96
BATCH_SIZE = 4


def sample_text(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    torch.manual_seed(2026)  # Reuse the sampling seed before and after.
    with torch.inference_mode():
        tokens = model.generate(
            **inputs, max_new_tokens=48, do_sample=True,
            temperature=0.85, top_p=0.92,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(tokens[0], skip_special_tokens=True)


def probe_loss(model, probe):
    model.eval()
    tokens, mask, targets = (tensor.to(DEVICE) for tensor in probe)
    with torch.no_grad():
        return model(
            input_ids=tokens, attention_mask=mask, labels=targets
        ).loss.item()


def run_experiment(track, count=64, steps=32):
    examples = training_examples(track, count)
    print(f"Track: {track} | {len(examples)} training examples")
    print("Example:", repr(examples[0][:180]))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.config.pad_token_id = tokenizer.pad_token_id

    encoded = tokenizer(
        examples, max_length=TOKEN_LIMIT, truncation=True,
        padding="max_length", return_tensors="pt",
    )
    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100  # Ignore padding in loss.
    loader = DataLoader(
        TensorDataset(encoded["input_ids"], encoded["attention_mask"], labels),
        batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )

    # Check both loss and a real parameter before updating anything.
    probe = tuple(tensor[:4] for tensor in (
        encoded["input_ids"], encoded["attention_mask"], labels
    ))
    loss_before = probe_loss(model, probe)
    weights_before = model.transformer.h[0].ln_1.weight.detach().cpu().clone()
    prompt = prompt_for(track)
    before_text = sample_text(model, tokenizer, prompt)

    optimizer = AdamW(model.parameters(), lr=1e-4)
    history = []
    batches = iter(loader)
    model.train()
    for step in range(1, steps + 1):
        try:
            tokens, mask, targets = next(batches)
        except StopIteration:
            batches = iter(loader)
            tokens, mask, targets = next(batches)

        tokens, mask, targets = (
            tensor.to(DEVICE) for tensor in (tokens, mask, targets)
        )
        optimizer.zero_grad(set_to_none=True)
        loss = model(
            input_ids=tokens, attention_mask=mask, labels=targets
        ).loss
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
        if step == 1 or step % 8 == 0 or step == steps:
            print(f"Step {step:>2}/{steps}: loss {history[-1]:.3f}")

    loss_after = probe_loss(model, probe)
    changed = (
        model.transformer.h[0].ln_1.weight.detach().cpu() - weights_before
    ).abs().max().item()
    after_text = sample_text(model, tokenizer, prompt)

    print(f"\nLoss on the same 4 training examples: {loss_before:.3f} → {loss_after:.3f}")
    print(f"Largest change to a sampled layer-norm weight: {changed:.7f}")
    print("\nBEFORE:\n", before_text)
    print("\nAFTER:\n", after_text)
    plt.figure(figsize=(7, 3))
    plt.plot(history)
    plt.xlabel("Training step")
    plt.ylabel("Next-token prediction loss")
    plt.title(f"GPT-2 fine-tuning: {track}")
    plt.show()
    if loss_after >= loss_before:
        print("This loss did not fall. Try more steps, or inspect the run!")
    return model, tokenizer, {"loss_before": loss_before, "loss_after": loss_after, "weight_change": changed}


### Go!

Run the next cell. The model will take 32 optimizer steps on a small batch of stories. You should see a loss curve, a before/after sample, and a check that its weights changed.


In [ ]:
baseline_model, baseline_tokenizer, baseline_result = run_experiment(
    "stories", count=64, steps=32
)


**You did it.** 🎉 You changed a transformer's weights.

One caveat worth keeping: **lower training loss isn't proof of generalization**. The text samples are playful illustrations, not a rigorous evaluation. That distinction is a good place to start asking questions.


## Part B — Make a choice

Don't just hit Run all again. **Check the box, choose stories or math, adjust how much to train, and press ▶ on the next cell.**

The second run loads a *fresh* copy of GPT-2, so your comparison won't accidentally keep training the first one. You can change the controls and rerun as often as you like.

What do you think will change? What would count as evidence that the model learned something beyond these examples?


In [ ]:
# @title 🎛️ My experiment (change the controls, then run this cell) { display-mode: "form" }
RUN_REMIX = False  # @param {type:"boolean"}
TRACK = "math"  # @param ["stories", "math"]
EXAMPLES = 64  # @param {type:"slider", min:16, max:96, step:16}
STEPS = 32  # @param {type:"slider", min:8, max:80, step:8}

if not RUN_REMIX:
    print("Your turn: check RUN_REMIX, choose a track, then press ▶ on this cell.")
else:
    # Make room on a free-tier GPU before loading another full model.
    if "baseline_model" in globals():
        baseline_model.to("cpu")
    if "remix_model" in globals():
        remix_model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    remix_model, remix_tokenizer, remix_result = run_experiment(
        TRACK, count=EXAMPLES, steps=STEPS
    )


## Your next question 🧭

- If you chose **math**, try changing the prompt to an addition problem the model didn't train on. Does it do better on familiar examples than unseen ones? A change in answer doesn't demonstrate arithmetic generalization.
- If you chose **stories**, change a character or setting in the prompt. What counts as an actual improvement, rather than a lucky sample?
- Try changing the **number of examples** or **training steps**. What do you predict? How would you test it?
- Ask someone else (or a coding assistant) to help you modify an experiment. You don't need to know the whole field to start exploring.

Want another rabbit hole? [**Pretrain a GPT-2 architecture from random weights**](https://colab.research.google.com/github/bubskee/hctigt/blob/main/notebooks/02-from-scratch.ipynb): choose a corpus and train it yourself. It is entirely optional — you've already completed the first step.

Want to build a transformer from its components? Try [ARENA](https://arena.education/).

*This notebook is an invitation to begin, not a benchmark or a research result.*
